# Transformer Multivariate

In this section we implement multivariate forecasting using the Transformer Model with the **TimeSeriesDatasetVectorizedExog** approach.

The Transformer Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate, ...).

With this approach the model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. The self-attention mechanism captures dependencies across the entire sequence simultaneously.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Input Projection (input_size → d_model)
    ↓
Positional Encoding
    ↓
Transformer Encoder Layers (2 layers)
  ├─ Multi-Head Self-Attention (4 heads)
  ├─ Add & Norm
  ├─ Feedforward (d_model → 256 → d_model)
  └─ Add & Norm
    ↓
Global Average Pooling
    ↓
Dropout
    ↓
Fully Connected (d_model → 1)
    ↓
Output (1 prediction)
```

## Model

In [ ]:
import torch 
import torch.nn as nn
import math

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding for Transformer model.
    Adds information about the position of tokens in the sequence.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, d_model)
        return x + self.pe[:, :x.size(1), :]


class TransformerForecaster(nn.Module):
    """
    Transformer model for MULTIVARIATE time series forecasting.
    Architecture: 
        Input Projection -> Positional Encoding -> 
        Transformer Encoder -> Global Average Pooling -> 
        Dropout -> Fully Connected
    
    Uses self-attention mechanism to capture dependencies.
    Can process entire sequence in parallel (unlike RNN/LSTM).
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=256, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            d_model: Dimension of the model (must be divisible by nhead)
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super(TransformerForecaster, self).__init__()
        
        self.input_size = input_size
        self.d_model = d_model
        
        # Input projection: map input_size to d_model
        self.input_projection = nn.Linear(input_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Important: batch_first=True for (batch, seq, feature) format
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Project input to d_model dimensions
        x = self.input_projection(x)  # (batch_size, seq_length, d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Pass through transformer encoder
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Global average pooling over sequence dimension
        # Alternative: use last token or first token (like BERT's [CLS])
        pooled = transformer_out.mean(dim=1)  # (batch_size, d_model)
        
        # Apply dropout
        out = self.dropout(pooled)
        
        # Fully connected layer
        out = self.fc(out)  # (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | d_model | Dim Feedforward | Dropout | Learning Rate | Nhead | Num Layers | Duration (s) |
|-------|----------------|------------|---------|-----------------|---------|---------------|-------|------------|--------------|
| 0 | 0.1348 | 4 | 64 | 256 | 0.2299 | 0.000471 | 4 | 3 | 7.41 |
| 1 | 0.1615 | 8 | 64 | 512 | 0.3749 | 0.000154 | 4 | 2 | 8.20 |
| 2 | 0.1633 | 8 | 64 | 256 | 0.3847 | 0.000239 | 8 | 2 | 5.10 |
| 3 | 0.1517 | 4 | 32 | 512 | 0.2443 | 0.005663 | 8 | 3 | 20.98 |
| 4 | 0.1689 | 16 | 32 | 256 | 0.2720 | 0.000113 | 2 | 1 | 5.41 |


### Best Hyperparameters

Parameters for Trial 0:
- learning_rate: 0.00047
- batch_size: 4
- d_model: 64
- nhead: 4
- num_layers: 3
- dim_feedforward: 256
- dropout: 0.229947


#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/transformers/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/transformers/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/transformers/fold3/fold_results.png)


### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|----------|----------|--------|-----------|
| Fold 1 | 74463.94 | 272.88 | 114.80 | 0.8524 | 83.17 |
| Fold 2 | 79190.88 | 281.41 | 114.12 | 0.8251 | 91.09 |
| Fold 3 | 69676.99 | 263.96 | 100.54 | 0.8567 | 72.45 |
| **Average** | **74443.94 ± 4756.98** | **272.75 ± 8.72** | **109.82 ± 8.05** | **0.8448 ± 0.0171** | **82.24 ± 9.35** |

### SMAPE Distribution Accross Folds 

| Metric | Value |
|--------|-------|
| MSE | 74443.94 ± 4756.98 |
| RMSE | 272.75 ± 8.72 |
| MAE | 109.82 ± 8.05 |
| R² | 0.8448 ± 0.0171 |
| SMAPE | 82.24% ± 9.35% |

**Comparison with Baseline:**

The Transformer multivariate model achieves an average SMAPE of 82.24% ± 9.35%, which is **9.98 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). The Transformer model underperforms compared to the baseline with slightly higher variation (standard deviation: 9.35% vs 7.06%). The model performs best in Fold 3 (72.45% SMAPE) but still lags behind the baseline. This suggests that despite the self-attention mechanism's ability to capture dependencies across the entire sequence, the Transformer architecture may require more data or different hyperparameters to effectively model the temporal patterns in this multivariate time series dataset. The parallel processing advantage of Transformers does not translate to better forecasting performance in this case.

## Model Results with Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Embedding Dimension | Dim Feedforward | Dropout | Learning Rate | Nhead | Num Layers | Duration (s) |
|-------|----------------|------------|---------|-----------------|---------|---------------|-------|------------|--------------|
| 0 | 0.4635 | 4 | 32 | 128 | 0.1193 | 0.000107 | 2 | 3 | 3.79 |
| 1 | 0.2103 | 8 | 128 | 512 | 0.3508 | 0.000276 | 2 | 3 | 39.75 |
| 2 | 0.2532 | 4 | 64 | 512 | 0.2625 | 0.004797 | 8 | 1 | 5.14 |
| 3 | 0.3374 | 4 | 128 | 128 | 0.3577 | 0.000769 | 4 | 3 | 7.72 |
| 4 | 0.5197 | 16 | 64 | 512 | 0.1802 | 0.000688 | 4 | 1 | 2.12 |

### Best Hyperparameters 

Parameters for Trial 1:
- learning_rate: 0.00027
- batch_size: 8
- d_model: 128
- nhead: 2
- num_layers: 3
- dim_feedforward: 512
- dropout: 0.3508




#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/transformers_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/transformers_exog/fold2/fold_results.png)


#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30
    
![Fold 3 Results](./img/multivariate/transformers_exog/fold3/fold_results.png)


### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|----------|----------|--------|-----------|
| Fold 1 | 255446.32 | 505.42 | 438.53 | 0.4938 | 136.59 |
| Fold 2 | 279882.45 | 529.04 | 483.49 | 0.3820 | 139.74 |
| Fold 3 | 86907.71 | 294.80 | 144.08 | 0.8213 | 114.67 |
| **Average** | **207412.16 ± 105072.71** | **443.09 ± 128.96** | **355.36 ± 184.35** | **0.5657 ± 0.2283** | **130.33 ± 13.65** |

| SMAPE Range | Average SMAPE | Std Dev | Number of Series |
|-------------|---------------|---------|------------------|
| <10% | 5.0% | ±7.6% | 76 |
| 10-20% | 2.9% | ±0.9% | 44 |
| 20-30% | 4.2% | ±0.3% | 64 |
| 30-40% | 4.6% | ±0.8% | 69 |
| >40% | 83.2% | ±8.6% | 1250 |

### SMAPE Distribution Across Folds

| Metric | Value |
|--------|-------|
| MSE | 207412.16 ± 105072.71 |
| RMSE | 443.09 ± 128.96 |
| MAE | 355.36 ± 184.35 |
| R² | 0.5657 ± 0.2283 |
| SMAPE | 130.33% ± 13.65% |

**Comparison with Baseline:**

The Transformer multivariate model with exogenous features achieves an average SMAPE of 130.33% ± 13.65%, which is **58.07 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). The model significantly underperforms compared to the baseline, showing much worse forecasting accuracy. Despite incorporating exogenous variables (GDP, CPI, Interest Rate), the model struggles to capture the temporal patterns effectively, with 1250 out of 1502 series (83.2%) having SMAPE values above 40%. The high variation across folds (std dev: 13.65%) and the dramatic performance difference between Fold 3 (114.67% SMAPE) and Folds 1-2 (~138% SMAPE) suggest instability in the model's predictions. This indicates that adding exogenous features actually degraded performance compared to the Transformer model without exogenous features (82.24% SMAPE), possibly due to the model's inability to effectively leverage the additional information in the limited training data available.
